In [1]:
from bs4 import BeautifulSoup
import aiohttp
from app.constants import constant
from app.schema.user_schema import UserLogin
from app import error_status
import json


async def login_using_reg_no_ums_home(user: UserLogin):
    url = constant.UMS_LOGIN_URL
    headers = constant.USER_AGENT_FORM_URL_ENCODED
    async with aiohttp.ClientSession() as session:
        async with session.get(
            url,
            headers=headers,
        ) as res:
            #! This will get us first event and view state
            html = await res.text()
            soup = BeautifulSoup(html, "html.parser")
            __LASTFOCUS = ""
            __EVENTTARGET = ""
            __EVENTARGUMENT = ""
            __VIEWSTATE = soup.find("input", {"id": "__VIEWSTATE"})["value"]
            __VIEWSTATEGENERATOR = soup.find("input", {"id": "__VIEWSTATEGENERATOR"})[
                "value"
            ]
            __SCROLLPOSITIONX = "0"
            __SCROLLPOSITIONY = "0"
            __EVENTVALIDATION = soup.find("input", {"id": "__EVENTVALIDATION"})["value"]
            txtU = user.reg_no
            TxtpwdAutoId_8767 = user.password
            DropDownList1 = "1"
            ddlStartWith = "StudentDashboard.aspx"
            iBtnLogins_x = "40"
            iBtnLogins_y = "50"

            #! Payload just with reg_number.
            payload_with_reg_no_only = {
                "__LASTFOCUS": __LASTFOCUS,
                "__EVENTTARGET": "txtU",
                "__EVENTARGUMENT": __EVENTARGUMENT,
                "__VIEWSTATE": (__VIEWSTATE),
                "__VIEWSTATEGENERATOR": __VIEWSTATEGENERATOR,
                "__SCROLLPOSITIONX": __SCROLLPOSITIONX,
                "__SCROLLPOSITIONY": __SCROLLPOSITIONY,
                "__EVENTVALIDATION": (__EVENTVALIDATION),
                "txtU": txtU,
                "TxtpwdAutoId_8767": "",
                "DropDownList1": DropDownList1,
            }
            soup.decompose()
            #! Here we will make a post request just with the user id and it will give us updated states
            async with session.post(
                url, headers=headers, data=payload_with_reg_no_only
            ) as res:
                html = await res.text()
                soup = BeautifulSoup(html, "html.parser")

                __VIEWSTATE = soup.find("input", {"id": "__VIEWSTATE"})["value"]
                __VIEWSTATEGENERATOR = soup.find(
                    "input", {"id": "__VIEWSTATEGENERATOR"}
                )["value"]
                __SCROLLPOSITIONX = "0"
                __SCROLLPOSITIONY = "0"
                __EVENTVALIDATION = soup.find("input", {"id": "__EVENTVALIDATION"})[
                    "value"
                ]

                #! Payload with updated state and Password
                payload = {
                    "__LASTFOCUS": __LASTFOCUS,
                    "__EVENTTARGET": __EVENTTARGET,
                    "__EVENTARGUMENT": __EVENTARGUMENT,
                    "__VIEWSTATE": (__VIEWSTATE),
                    "__VIEWSTATEGENERATOR": __VIEWSTATEGENERATOR,
                    "__SCROLLPOSITIONX": __SCROLLPOSITIONX,
                    "__SCROLLPOSITIONY": __SCROLLPOSITIONY,
                    "__EVENTVALIDATION": (__EVENTVALIDATION),
                    "txtU": txtU,
                    "TxtpwdAutoId_8767": TxtpwdAutoId_8767,
                    "ddlStartWith": ddlStartWith,
                    "iBtnLogins150203125": "Login",
                    # "iBtnLogins.x": iBtnLogins_x,
                    # "iBtnLogins.y": iBtnLogins_y,
                }
                soup.decompose()
                #! SignIn with updated payload
                async with session.post(url, headers=headers, data=payload) as res:
                    asp_cookie = None
                    try:
                        asp_cookie = res.request_info.headers["Cookie"]
                    except:
                        await session.close()
                        raise error_status.SOMETHING_WRONG_WITH_UMS_SERVER
                    await session.close()
                    is_auth = await check_auth_status_ums_home(asp_cookie)
                    if is_auth:
                        return asp_cookie
                    raise error_status.CREDENTIALS_NOT_VALID

async def check_auth_status_ums_home(cookie) -> bool:
    async with aiohttp.ClientSession() as session:
        headers = constant.USER_AGENT_JSON
        headers["Cookie"] = cookie
        resp = await session.post(
            constant.UMS_STUDENT_PHONE_NUMBER_URL,
            headers=headers,
            data=json.dumps({}),
        )
        rs_json = await resp.json()
        await session.close()
        if rs_json.get("d", {}) is None:
            return False
        return True


In [9]:
from attendance_summary import get_attendance_summary, get_attendance_detail
from get_time_table import get_time_table_details

In [3]:
import nest_asyncio
nest_asyncio.apply()

In [11]:
akhil = UserLogin(reg_no="12309014", password="@Ak6918g")
cookies = await (login_using_reg_no_ums_home(akhil))
timetable = await get_time_table_details(cookies)
print(timetable)

{'time_table': {'Monday': {'09-10 AM': 'Lecture / G:All C:MTH302 / R: 33-605 / S:K23WT', '10-11 AM': 'Lecture / G:All C:INT255 / R: 33-605 / S:K23WT', '11-12 AM': 'Lecture / G:All C:INT255 / R: 33-605 / S:K23WT', '12-01 PM': 'Lecture / G:All C:CSE211 / R: 33-605 / S:K23WT', '01-02 PM': '', '02-03 PM': '', '03-04 PM': '', '04-05 PM': ''}, 'Tuesday': {'09-10 AM': 'Tutorial / G:1 C:PEA305 / R: 34-704 / S:K23WT', '10-11 AM': 'Practical / G:0 C:CSE310 / R: 28-507A / S:K23WT', '11-12 AM': 'Practical / G:0 C:CSE310 / R: 28-507A / S:K23WT', '12-01 PM': '', '01-02 PM': 'Lecture / G:All C:INT256 / R: 28-507A / S:K23WT', '02-03 PM': 'Lecture / G:All C:INT256 / R: 28-507A / S:K23WT', '03-04 PM': 'Lecture / G:All C:PEA305 / R: 28-507A / S:K23WT', '04-05 PM': 'Tutorial / G:1 C:MTH302 / R: 34-704 / S:K23WT'}, 'Wednesday': {'09-10 AM': '', '10-11 AM': 'Lecture / G:All C:MTH302 / R: 28-507 / S:K23WT', '11-12 AM': 'Lecture / G:All C:CSE211 / R: 28-507 / S:K23WT', '12-01 PM': '', '01-02 PM': 'Lecture / G